# Run 5: Bone Vectors, BBox Normalization, Attention Pooling

Run 4 established the sequential ablation pipeline: A (skeleton repr) → B (temporal window) → C (shuttle fusion) → D (classifier head). This run inserts **two new groups** into the pipeline at the correct positions:

- **Group A+ : Feature representation** — bone vectors and bbox normalization. These are skeleton-level input features (same level as A's L2/L3/hitter choices), tested on top of A1 (dual-player L2 baseline, the A winner).
- **Group B+ : Pooling strategy** — attention pooling vs mean pooling. This is an encoder architecture change, tested with the A+ winner before shuttle fusion.
- **Group C+ : Full cascade** — A+ winner + B+ winner + shuttle cross-attention (re-running C with the improved encoder).

The cascade flows: **A1 → A+ → B+ → C+**, matching the original dependency logic.

**Why these changes:**
- Bone vectors encode limb direction/length explicitly — elbow-high (full smash) vs elbow-low (tap smash)
- BBox normalization makes skeleton scale-invariant (camera distance variation)
- Attention pooling lets the model weight contact frames and hitting-arm joints instead of averaging everything equally
- These address the two BST features we were missing (bones, bbox-norm) and the mean-pooling bottleneck identified in run4 analysis

In [1]:
import os, sys, json, copy, time
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT   = Path('/content/drive/MyDrive/Baddiev2')
    PROJECT_PATH = Path('/content/Baddiev2')

    import zipfile
    ZIP_PATH = DRIVE_ROOT / 'baddiev2_colab.zip'
    if not (PROJECT_PATH / 'src').exists():
        print('Extracting project files...')
        with zipfile.ZipFile(ZIP_PATH, 'r') as z:
            z.extractall(PROJECT_PATH)
    sys.path.insert(0, str(PROJECT_PATH))
    os.chdir(PROJECT_PATH)

    import src.config as _cfg
    _cfg.MODELS_DIR            = DRIVE_ROOT / 'models'
    _cfg.RESULTS_DIR           = DRIVE_ROOT / 'results'
    _cfg.SS_SKELETONS_GDINO    = DRIVE_ROOT / 'datasets_preprocessing' / 'shuttleset_skeletons_gdino'
    _cfg.SS_SHUTTLES           = DRIVE_ROOT / 'datasets_preprocessing' / 'shuttleset_shuttles'
    _cfg.FB_ANNOTATIONS        = (
        DRIVE_ROOT / 'datasets' / 'FineBadminton-dataset' / 'dataset' /
        'transformed_combined_rounds_output_en_evals_translated.json'
    )
    _cfg.SS_CSV_ROOT   = DRIVE_ROOT / 'datasets' / 'ShuttleSet' / 'set'
    _cfg.SS_MATCH_CSV  = _cfg.SS_CSV_ROOT / 'match.csv'
    _cfg.SS_SPLIT_JSON = PROJECT_PATH / 'datasets_preprocessing' / 'shuttleset_split.json'
    _cfg.MODELS_DIR.mkdir(parents=True, exist_ok=True)
    _cfg.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    print(f'Drive root: {DRIVE_ROOT}')
else:
    sys.path.insert(0, os.path.abspath('../..'))
    DRIVE_ROOT = Path('../..')
    print('Local run')

Local run


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset as _Dataset
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.metrics import f1_score, classification_report, accuracy_score, confusion_matrix

from src.config import (
    get_config, FEATURE_DIMS, FEATURE_DIMS_WITH_HITTER, BONE_CHANNELS,
    SS_SKELETONS_GDINO, SS_SHUTTLES, SS_SPLIT_JSON,
    SHOT_TYPES, NUM_SHOT_TYPES,
    NUM_NODES, NUM_JOINTS, MODELS_DIR, RESULTS_DIR,
)
from src.data.graph_builder import GraphBuilder
from src.data.dataset import ShuttleSetDataset
from src.models.stgcn_model import STGCN
from src.models.shuttle_cross_attn import ShuttleCrossAttention, ShuttleTCN

device = torch.device(
    'cuda' if torch.cuda.is_available() else
    'mps'  if torch.backends.mps.is_available() else
    'cpu'
)
print(f'Device: {device}')

ABLATION_DIR = RESULTS_DIR / 'ablations'
ABLATION_DIR.mkdir(parents=True, exist_ok=True)
print(f'Ablation results: {ABLATION_DIR}')

Device: mps
Ablation results: /Users/yuen@backbase.com/Documents/Baddiev2/results/ablations


In [3]:
# Copy data to local SSD on Colab for faster I/O
if IN_COLAB:
    import shutil
    local_skel = Path('/content/local_skeletons')
    local_shuttle = Path('/content/local_shuttles')

    if not local_skel.exists():
        print('Copying skeletons to local SSD...')
        t0 = time.time()
        shutil.copytree(SS_SKELETONS_GDINO, local_skel)
        print(f'Done in {time.time()-t0:.0f}s')

    if not local_shuttle.exists():
        print('Copying shuttles to local SSD...')
        t0 = time.time()
        shutil.copytree(SS_SHUTTLES, local_shuttle)
        print(f'Done in {time.time()-t0:.0f}s')
else:
    local_skel = SS_SKELETONS_GDINO
    local_shuttle = SS_SHUTTLES

## 1. Shared Setup

In [4]:
# ── Reproducibility ───────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True

# ── Shared hyperparameters (same as run4) ─────────────────────────────────
SHOT_WINDOW   = 32
EPOCHS        = 80
BATCH_SIZE    = 64
LR            = 1e-3
WEIGHT_DECAY  = 1e-4
PATIENCE      = 10
N_CLASSES     = NUM_SHOT_TYPES  # 17

# ── Train / val / test split (same as run4) ──────────────────────────────
with open(SS_SPLIT_JSON) as f:
    splits = json.load(f)

VAL_MATCH_NAMES = [
    'NG_Ka_Long_Angus_Jonatan_CHRISTIE_Malaysia_Masters_2020_QuarterFinals',
    'Anders_Antonsen_Sameer_Verma_TOYOTA_THAILAND_OPEN_2021_QuarterFinals',
]
TRAIN_MATCHES = set(splits['train']) - set(VAL_MATCH_NAMES)
VAL_MATCHES   = set(VAL_MATCH_NAMES)
TEST_MATCHES  = set(splits['held_out'])
print(f'Train: {len(TRAIN_MATCHES)} matches | Val: {len(VAL_MATCHES)} matches | Test: {len(TEST_MATCHES)} matches')

Train: 17 matches | Val: 2 matches | Test: 2 matches


## 2. Utility Functions

In [5]:
def make_dataset(split_matches, feature_layer='L2', use_hitter=False,
                 use_shuttle=False, shuttle_fusion='graph',
                 use_bones=False, use_bbox_norm=False):
    """Build a ShuttleSetDataset filtered to the given match set."""
    ds = ShuttleSetDataset(
        skeleton_dir=local_skel,
        shot_window=SHOT_WINDOW,
        feature_layer=feature_layer,
        load_shot_types=True,
        split=None,
        use_shuttle=use_shuttle,
        shuttle_dir=local_shuttle,
        use_hitter=use_hitter,
        shuttle_fusion=shuttle_fusion,
        use_bones=use_bones,
        use_bbox_norm=use_bbox_norm,
    )
    ds.samples = [s for s in ds.samples
                  if isinstance(s, dict) and Path(s.get('skel_dir', '')).name in split_matches]
    n_valid = sum(1 for s in ds.samples if s.get('shot_type_idx') is not None)
    print(f'  {len(ds)} samples ({n_valid} with labels)')
    return ds


def collate_fn(batch):
    xs, labels = [], []
    for item in batch:
        xs.append(item[0])
        labels.append(item[1])
    return torch.stack(xs), torch.tensor(labels, dtype=torch.long)


def collate_fn_shuttle(batch):
    xs, labels, shuttles = [], [], []
    for item in batch:
        xs.append(item[0])
        labels.append(item[1])
        if len(item) == 3:
            shuttles.append(item[2])
        else:
            shuttles.append(torch.zeros(2, SHOT_WINDOW))
    return torch.stack(xs), torch.tensor(labels, dtype=torch.long), torch.stack(shuttles)


def compute_class_weights(dataset):
    from collections import Counter
    labels = [s.get('shot_type_idx') for s in dataset.samples
              if s.get('shot_type_idx') is not None]
    counts = Counter(labels)
    total = sum(counts.values())
    weights = torch.ones(N_CLASSES, dtype=torch.float32)
    for cls_id, cnt in counts.items():
        weights[cls_id] = total / (len(counts) * cnt)
    return weights


def build_encoder(in_channels, num_nodes, use_inter_player=True, pooling='mean'):
    """Build ST-GCN encoder + adjacency for the given config."""
    graph = GraphBuilder(use_inter_player=use_inter_player)
    adj = graph.build_adjacency().to(device)
    encoder = STGCN(
        in_channels=in_channels,
        num_nodes=num_nodes,
        adjacency=adj,
        num_layers=9,
        base_channels=64,
        embedding_dim=256,
        temporal_kernel=9,
        dropout=0.3,
        pooling=pooling,
    ).to(device)
    return encoder

In [6]:
def evaluate(encoder, head, ds, cross_attn_module=None):
    """Run batched evaluation, return (macro_f1, accuracy, y_true, y_pred, topk_metrics)."""
    use_cross_attn = cross_attn_module is not None
    cfn = collate_fn_shuttle if use_cross_attn else collate_fn

    loader = DataLoader(
        ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=0, pin_memory=True, collate_fn=cfn,
    )

    encoder.eval(); head.eval()
    if use_cross_attn:
        cross_attn_module.eval()

    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            if use_cross_attn:
                xb, yb, shuttle_b = batch
                shuttle_b = shuttle_b.to(device)
            else:
                xb, yb = batch

            valid = yb >= 0
            if not valid.any():
                continue

            xb_v = xb[valid].to(device)
            emb = encoder(xb_v)

            if use_cross_attn:
                emb = cross_attn_module(emb, shuttle_b[valid].to(device))

            logits = head(emb)
            all_preds.append(logits.cpu())
            all_labels.append(yb[valid])

    if not all_preds:
        return 0.0, 0.0, np.array([]), np.array([]), {}

    all_logits = torch.cat(all_preds)
    y_true = torch.cat(all_labels).numpy()
    y_pred = all_logits.argmax(dim=1).numpy()
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    accuracy = accuracy_score(y_true, y_pred)

    topk_metrics = {}
    y_true_t = torch.tensor(y_true, dtype=torch.long)
    for k in [3, 5]:
        if all_logits.shape[1] >= k:
            topk_preds = all_logits.topk(k, dim=1).indices
            correct = topk_preds.eq(y_true_t.unsqueeze(1)).any(dim=1)
            topk_metrics[f'top{k}_acc'] = correct.float().mean().item()

    return macro_f1, accuracy, y_true, y_pred, topk_metrics

In [7]:
def train_and_evaluate(name, train_ds, val_ds, test_ds, encoder, head,
                       cross_attn_module=None, epochs=EPOCHS):
    """
    Train encoder + head with CE loss. Early stop on val_ds macro-F1.
    Final evaluation on test_ds (never used for model selection).
    """
    use_cross_attn = cross_attn_module is not None
    cfn = collate_fn_shuttle if use_cross_attn else collate_fn

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=0, pin_memory=True, drop_last=True, collate_fn=cfn,
    )

    class_weights = compute_class_weights(train_ds).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    params = list(encoder.parameters()) + list(head.parameters())
    if use_cross_attn:
        params += list(cross_attn_module.parameters())

    optimizer = optim.AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_val_f1 = -1.0
    no_improve = 0
    best_state = None
    history = {'train_loss': [], 'val_f1': [], 'val_acc': []}
    t0 = time.time()

    for epoch in range(epochs):
        # ── Train ─────────────────────────────────────────────────────
        encoder.train()
        head.train()
        if use_cross_attn:
            cross_attn_module.train()
        epoch_loss = 0.0
        n_batches = 0

        for batch in tqdm(train_loader, desc=f'[{name}] Epoch {epoch+1}/{epochs}', leave=False):
            if use_cross_attn:
                xb, yb, shuttle_b = batch
                shuttle_b = shuttle_b.to(device)
            else:
                xb, yb = batch

            valid = yb >= 0
            if not valid.any():
                continue

            xb = xb[valid].to(device)
            yb = yb[valid].to(device)

            emb = encoder(xb)

            if use_cross_attn:
                shuttle_b = shuttle_b[valid]
                emb = cross_attn_module(emb, shuttle_b)

            logits = head(emb)
            loss = criterion(logits, yb)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(params, max_norm=1.0)
            optimizer.step()

            epoch_loss += loss.item()
            n_batches += 1

        scheduler.step()
        avg_loss = epoch_loss / max(n_batches, 1)

        # ── Validate ──────────────────────────────────────────────────
        val_f1, val_acc, _, _, _ = evaluate(encoder, head, val_ds, cross_attn_module)

        history['train_loss'].append(avg_loss)
        history['val_f1'].append(val_f1)
        history['val_acc'].append(val_acc)

        if (epoch + 1) % 10 == 0:
            print(f'  Epoch {epoch+1:3d}/{epochs} | loss: {avg_loss:.4f} | val_f1: {val_f1:.4f}')

        if val_f1 > best_val_f1 + 1e-4:
            best_val_f1 = val_f1
            no_improve = 0
            state = {
                'encoder': {k: v.cpu().clone() for k, v in encoder.state_dict().items()},
                'head': {k: v.cpu().clone() for k, v in head.state_dict().items()},
            }
            if use_cross_attn:
                state['cross_attn'] = {k: v.cpu().clone()
                                       for k, v in cross_attn_module.state_dict().items()}
            best_state = state
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print(f'  Early stopping at epoch {epoch+1} (best val F1: {best_val_f1:.4f})')
                break

    train_time = time.time() - t0

    # ── Restore best checkpoint ───────────────────────────────────────
    if best_state:
        encoder.load_state_dict(best_state['encoder'])
        head.load_state_dict(best_state['head'])
        if use_cross_attn and 'cross_attn' in best_state:
            cross_attn_module.load_state_dict(best_state['cross_attn'])

    # ── Final evaluation on TEST set ──────────────────────────────────
    macro_f1, accuracy, y_true, y_pred, topk = evaluate(encoder, head, test_ds, cross_attn_module)
    stopped_epoch = len(history['train_loss'])

    present_labels = sorted(set(y_true) | set(y_pred))
    label_names = [SHOT_TYPES[i] if i < len(SHOT_TYPES)
                   else f'type_{i}' for i in present_labels]

    print(f'\n  === {name} ====')
    print(f'  Test Accuracy:  {accuracy:.4f}')
    print(f'  Test Macro-F1:  {macro_f1:.4f}')
    top3 = topk.get('top3_acc', 0)
    top5 = topk.get('top5_acc', 0)
    print(f'  Top-3 Accuracy: {top3:.4f}')
    print(f'  Top-5 Accuracy: {top5:.4f}')
    print(f'  Best Val F1:    {best_val_f1:.4f}')
    print(f'  Epochs:         {stopped_epoch}')
    print(f'  Time:           {train_time:.0f}s')
    print(classification_report(
        y_true, y_pred, labels=present_labels,
        target_names=label_names, zero_division=0,
    ))

    report = classification_report(
        y_true, y_pred, labels=present_labels,
        target_names=label_names, zero_division=0, output_dict=True,
    )

    result = {
        'name': name,
        'accuracy': round(accuracy, 4),
        'macro_f1': round(macro_f1, 4),
        'top3_acc': round(topk.get('top3_acc', 0), 4),
        'top5_acc': round(topk.get('top5_acc', 0), 4),
        'best_val_f1': round(best_val_f1, 4),
        'stopped_epoch': stopped_epoch,
        'train_time_s': round(train_time, 1),
        'n_train': int(len(train_ds)),
        'n_val': int(len(val_ds)),
        'n_test': int(len(test_ds)),
        'n_test_labeled': int(len(y_true)),
        'history': history,
        'per_class': report,
    }
    save_path = ABLATION_DIR / f'{name}.json'
    with open(save_path, 'w') as f:
        json.dump(result, f, indent=2)
    print(f'  Saved: {save_path}')

    ckpt_path = MODELS_DIR / f'ablation_{name}.pt'
    ckpt = {
        'encoder_state_dict': encoder.state_dict(),
        'head_state_dict': head.state_dict(),
        'name': name,
        'accuracy': accuracy,
        'macro_f1': macro_f1,
    }
    if use_cross_attn:
        ckpt['cross_attn_state_dict'] = cross_attn_module.state_dict()
    torch.save(ckpt, ckpt_path)
    print(f'  Checkpoint: {ckpt_path}')

    return result

## 3. Group A+: Feature Representation (builds on A1 winner)

Run 4 Group A tested skeleton repr: L2 vs L3, single vs dual, hitter flag.
Winner was A1 (dual-player, L2, 9-dim). These ablations extend that with
BST-inspired features, **without shuttle fusion** — isolating the feature contribution.

| Ablation | Description | In Channels |
|----------|-------------|-------------|
| A7 (baseline) | A1 reproduction: L2, no bones, no bbox | 9 |
| A8 | + bone vectors | 11 |
| A9 | + bbox normalization | 9 |
| A10 | + bones + bbox | 11 |

In [8]:
# ── A7: Baseline (A1 reproduction for this run) ───────────────────────────
print('A7: Dual-player, L2 (baseline — same as run4 A1, no shuttle)')
train_A7 = make_dataset(TRAIN_MATCHES, feature_layer='L2')
val_A7   = make_dataset(VAL_MATCHES, feature_layer='L2')
test_A7  = make_dataset(TEST_MATCHES, feature_layer='L2')

in_ch_A7 = FEATURE_DIMS['L2']  # 9
encoder_A7 = build_encoder(in_channels=in_ch_A7, num_nodes=NUM_NODES, pooling='mean')
head_A7 = nn.Linear(256, N_CLASSES).to(device)
print(f'Parameters: {sum(p.numel() for p in encoder_A7.parameters()) + sum(p.numel() for p in head_A7.parameters()):,}')

result_A7 = train_and_evaluate('A7_baseline', train_A7, val_A7, test_A7,
                                encoder_A7, head_A7)

A7: Dual-player, L2 (baseline — same as run4 A1, no shuttle)
[INFO] Loaded SS homographies from ss_per_match_H.npy (37 matches)
[INFO] ShuttleSet: 37 per-match homographies
[INFO] ShuttleSet split='None': 17754 shots from whole-match skeletons across 21 match(es)
  13924 samples (13270 with labels)
[INFO] Loaded SS homographies from ss_per_match_H.npy (37 matches)
[INFO] ShuttleSet: 37 per-match homographies
[INFO] ShuttleSet split='None': 17754 shots from whole-match skeletons across 21 match(es)
  2155 samples (2077 with labels)
[INFO] Loaded SS homographies from ss_per_match_H.npy (37 matches)
[INFO] ShuttleSet: 37 per-match homographies
[INFO] ShuttleSet split='None': 17754 shots from whole-match skeletons across 21 match(es)
  1675 samples (1627 with labels)
Parameters: 3,088,149


[A7_baseline] Epoch 1/80:   0%|          | 0/217 [00:00<?, ?it/s]/Users/yuen@backbase.com/Documents/Baddiev2/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


KeyboardInterrupt: 

In [ ]:
# ── A8: Bone vectors only ─────────────────────────────────────────────────
print('A8: L2 + bone vectors')
train_A8 = make_dataset(TRAIN_MATCHES, feature_layer='L2', use_bones=True)
val_A8   = make_dataset(VAL_MATCHES, feature_layer='L2', use_bones=True)
test_A8  = make_dataset(TEST_MATCHES, feature_layer='L2', use_bones=True)

in_ch_A8 = FEATURE_DIMS['L2'] + BONE_CHANNELS  # 9 + 2 = 11
encoder_A8 = build_encoder(in_channels=in_ch_A8, num_nodes=NUM_NODES, pooling='mean')
head_A8 = nn.Linear(256, N_CLASSES).to(device)
print(f'Parameters: {sum(p.numel() for p in encoder_A8.parameters()) + sum(p.numel() for p in head_A8.parameters()):,}')

result_A8 = train_and_evaluate('A8_bones', train_A8, val_A8, test_A8,
                                encoder_A8, head_A8)

A8: L2 + bone vectors
[INFO] Loaded SS homographies from ss_per_match_H.npy (37 matches)
[INFO] ShuttleSet: 37 per-match homographies
[INFO] ShuttleSet split='None': 17754 shots from whole-match skeletons across 21 match(es)
  13924 samples (13270 with labels)
[INFO] Loaded SS homographies from ss_per_match_H.npy (37 matches)
[INFO] ShuttleSet: 37 per-match homographies
[INFO] ShuttleSet split='None': 17754 shots from whole-match skeletons across 21 match(es)
  2155 samples (2077 with labels)
[INFO] Loaded SS homographies from ss_per_match_H.npy (37 matches)
[INFO] ShuttleSet: 37 per-match homographies
[INFO] ShuttleSet split='None': 17754 shots from whole-match skeletons across 21 match(es)
  1675 samples (1627 with labels)
Parameters: 3,088,797


[A8_bones] Epoch 1/80:   0%|          | 0/217 [00:00<?, ?it/s]/Users/yuen@backbase.com/Documents/Baddiev2/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
[A8_bones] Epoch 2/80:   0%|          | 0/217 [00:00<?, ?it/s]          /Users/yuen@backbase.com/Documents/Baddiev2/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
[A8_bones] Epoch 3/80:   0%|          | 0/217 [00:00<?, ?it/s]          /Users/yuen@backbase.com/Documents/Baddiev2/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
[A8_bones] Epoch 4/8

  Epoch  10/80 | loss: 0.9156 | val_f1: 0.4654


[A8_bones] Epoch 11/80:   0%|          | 0/217 [00:00<?, ?it/s]/Users/yuen@backbase.com/Documents/Baddiev2/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
[A8_bones] Epoch 12/80:   0%|          | 0/217 [00:00<?, ?it/s]          /Users/yuen@backbase.com/Documents/Baddiev2/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
[A8_bones] Epoch 13/80:   0%|          | 0/217 [00:00<?, ?it/s]          /Users/yuen@backbase.com/Documents/Baddiev2/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
[A8_bones] Epoch 

  Epoch  20/80 | loss: 0.5029 | val_f1: 0.4645


...

  Epoch  30/80 | loss: 0.2715 | val_f1: 0.5174


[A8_bones] Epoch 31/80:   0%|          | 0/217 [00:00<?, ?it/s]/Users/yuen@backbase.com/Documents/Baddiev2/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
[A8_bones] Epoch 32/80:   0%|          | 0/217 [00:00<?, ?it/s]          /Users/yuen@backbase.com/Documents/Baddiev2/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
[A8_bones] Epoch 33/80:   0%|          | 0/217 [00:00<?, ?it/s]          /Users/yuen@backbase.com/Documents/Baddiev2/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


  Early stopping at epoch 33 (best val F1: 0.5276)


/Users/yuen@backbase.com/Documents/Baddiev2/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)



  === A8_bones ====
  Test Accuracy:  0.6398
  Test Macro-F1:  0.4904
  Top-3 Accuracy: 0.8937
  Top-5 Accuracy: 0.9490
  Best Val F1:    0.5276
  Epochs:         33
  Time:           9675s
                        precision    recall  f1-score   support

              net shot       0.80      0.84      0.82       296
            return net       0.78      0.66      0.72       163
                 smash       0.70      0.18      0.28       107
           wrist smash       0.35      0.67      0.46        75
                   lob       0.46      0.88      0.61       137
  defensive return lob       0.14      0.38      0.21         8
                 clear       0.93      0.90      0.91       129
                 drive       0.50      0.19      0.27        53
         driven flight       0.00      0.00      0.00         1
      back-court drive       0.50      0.16      0.24        25
                  drop       0.50      0.70      0.59        61
          passive drop       0.63      0

In [ ]:
# ── A9: BBox normalization only ───────────────────────────────────────────
print('A9: L2 + bbox normalization')
train_A9 = make_dataset(TRAIN_MATCHES, feature_layer='L2', use_bbox_norm=True)
val_A9   = make_dataset(VAL_MATCHES, feature_layer='L2', use_bbox_norm=True)
test_A9  = make_dataset(TEST_MATCHES, feature_layer='L2', use_bbox_norm=True)

in_ch_A9 = FEATURE_DIMS['L2']  # 9 (same dims, just normalized coords)
encoder_A9 = build_encoder(in_channels=in_ch_A9, num_nodes=NUM_NODES, pooling='mean')
head_A9 = nn.Linear(256, N_CLASSES).to(device)
print(f'Parameters: {sum(p.numel() for p in encoder_A9.parameters()) + sum(p.numel() for p in head_A9.parameters()):,}')

result_A9 = train_and_evaluate('A9_bbox_norm', train_A9, val_A9, test_A9,
                                encoder_A9, head_A9)

A9: L2 + bbox normalization
[INFO] Loaded SS homographies from ss_per_match_H.npy (37 matches)
[INFO] ShuttleSet: 37 per-match homographies
[INFO] ShuttleSet split='None': 17754 shots from whole-match skeletons across 21 match(es)
  13924 samples (13270 with labels)
[INFO] Loaded SS homographies from ss_per_match_H.npy (37 matches)
[INFO] ShuttleSet: 37 per-match homographies
[INFO] ShuttleSet split='None': 17754 shots from whole-match skeletons across 21 match(es)
  2155 samples (2077 with labels)
[INFO] Loaded SS homographies from ss_per_match_H.npy (37 matches)
[INFO] ShuttleSet: 37 per-match homographies
[INFO] ShuttleSet split='None': 17754 shots from whole-match skeletons across 21 match(es)
  1675 samples (1627 with labels)
Parameters: 3,088,149


[A9_bbox_norm] Epoch 1/80:   0%|          | 0/217 [00:00<?, ?it/s]/Users/yuen@backbase.com/Documents/Baddiev2/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
[A9_bbox_norm] Epoch 2/80:   0%|          | 0/217 [00:00<?, ?it/s]          /Users/yuen@backbase.com/Documents/Baddiev2/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
[A9_bbox_norm] Epoch 3/80:   0%|          | 0/217 [00:00<?, ?it/s]          /Users/yuen@backbase.com/Documents/Baddiev2/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
[A9_bbox

  Epoch  10/80 | loss: 1.0877 | val_f1: 0.2969


[A9_bbox_norm] Epoch 11/80:   0%|          | 0/217 [00:00<?, ?it/s]/Users/yuen@backbase.com/Documents/Baddiev2/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
[A9_bbox_norm] Epoch 12/80:   0%|          | 0/217 [00:00<?, ?it/s]          /Users/yuen@backbase.com/Documents/Baddiev2/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
[A9_bbox_norm] Epoch 13/80:   0%|          | 0/217 [00:00<?, ?it/s]          /Users/yuen@backbase.com/Documents/Baddiev2/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
[A9_b

  Epoch  20/80 | loss: 0.5970 | val_f1: 0.4603


...

  Epoch  40/80 | loss: 0.1608 | val_f1: 0.4879


[A9_bbox_norm] Epoch 41/80:   0%|          | 0/217 [00:00<?, ?it/s]/Users/yuen@backbase.com/Documents/Baddiev2/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
[A9_bbox_norm] Epoch 42/80:   0%|          | 0/217 [00:00<?, ?it/s]          /Users/yuen@backbase.com/Documents/Baddiev2/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
[A9_bbox_norm] Epoch 43/80:   0%|          | 0/217 [00:00<?, ?it/s]          /Users/yuen@backbase.com/Documents/Baddiev2/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
[A9_b

  Early stopping at epoch 45 (best val F1: 0.5083)


/Users/yuen@backbase.com/Documents/Baddiev2/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)



  === A9_bbox_norm ====
  Test Accuracy:  0.5759
  Test Macro-F1:  0.4487
  Top-3 Accuracy: 0.8826
  Top-5 Accuracy: 0.9299
  Best Val F1:    0.5083
  Epochs:         45
  Time:           13994s
                        precision    recall  f1-score   support

              net shot       0.73      0.82      0.78       296
            return net       0.71      0.55      0.62       163
                 smash       0.51      0.36      0.42       107
           wrist smash       0.33      0.32      0.33        75
                   lob       0.40      0.86      0.55       137
  defensive return lob       0.30      0.38      0.33         8
                 clear       0.90      0.87      0.89       129
                 drive       0.44      0.13      0.20        53
         driven flight       0.00      0.00      0.00         1
      back-court drive       0.21      0.12      0.15        25
                  drop       0.29      0.66      0.41        61
          passive drop       0.52  

In [9]:
# ── A10: Bones + BBox normalization ───────────────────────────────────────
print('A10: L2 + bones + bbox norm')
train_A10 = make_dataset(TRAIN_MATCHES, feature_layer='L2',
                         use_bones=True, use_bbox_norm=True)
val_A10   = make_dataset(VAL_MATCHES, feature_layer='L2',
                         use_bones=True, use_bbox_norm=True)
test_A10  = make_dataset(TEST_MATCHES, feature_layer='L2',
                         use_bones=True, use_bbox_norm=True)

in_ch_A10 = FEATURE_DIMS['L2'] + BONE_CHANNELS  # 11
encoder_A10 = build_encoder(in_channels=in_ch_A10, num_nodes=NUM_NODES, pooling='mean')
head_A10 = nn.Linear(256, N_CLASSES).to(device)
print(f'Parameters: {sum(p.numel() for p in encoder_A10.parameters()) + sum(p.numel() for p in head_A10.parameters()):,}')

result_A10 = train_and_evaluate('A10_bones_bbox', train_A10, val_A10, test_A10,
                                 encoder_A10, head_A10)

A10: L2 + bones + bbox norm
[INFO] Loaded SS homographies from ss_per_match_H.npy (37 matches)
[INFO] ShuttleSet: 37 per-match homographies
[INFO] ShuttleSet split='None': 17754 shots from whole-match skeletons across 21 match(es)
  13924 samples (13270 with labels)
[INFO] Loaded SS homographies from ss_per_match_H.npy (37 matches)
[INFO] ShuttleSet: 37 per-match homographies
[INFO] ShuttleSet split='None': 17754 shots from whole-match skeletons across 21 match(es)
  2155 samples (2077 with labels)
[INFO] Loaded SS homographies from ss_per_match_H.npy (37 matches)
[INFO] ShuttleSet: 37 per-match homographies
[INFO] ShuttleSet split='None': 17754 shots from whole-match skeletons across 21 match(es)
  1675 samples (1627 with labels)
Parameters: 3,088,797


[A10_bones_bbox] Epoch 2/80:   0%|          | 0/217 [00:00<?, ?it/s]          /Users/yuen@backbase.com/Documents/Baddiev2/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
[A10_bones_bbox] Epoch 3/80:   0%|          | 0/217 [00:00<?, ?it/s]          /Users/yuen@backbase.com/Documents/Baddiev2/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
[A10_bones_bbox] Epoch 4/80:   0%|          | 0/217 [00:00<?, ?it/s]          /Users/yuen@backbase.com/Documents/Baddiev2/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(

  Epoch  10/80 | loss: 1.0603 | val_f1: 0.4433


[A10_bones_bbox] Epoch 11/80:   0%|          | 0/217 [00:00<?, ?it/s]/Users/yuen@backbase.com/Documents/Baddiev2/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
[A10_bones_bbox] Epoch 12/80:   0%|          | 0/217 [00:00<?, ?it/s]          /Users/yuen@backbase.com/Documents/Baddiev2/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
[A10_bones_bbox] Epoch 13/80:   0%|          | 0/217 [00:00<?, ?it/s]          /Users/yuen@backbase.com/Documents/Baddiev2/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)

  Epoch  20/80 | loss: 0.5787 | val_f1: 0.4705


...

  Epoch  30/80 | loss: 0.3247 | val_f1: 0.4688


[A10_bones_bbox] Epoch 31/80:   0%|          | 0/217 [00:00<?, ?it/s]/Users/yuen@backbase.com/Documents/Baddiev2/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
[A10_bones_bbox] Epoch 32/80:   0%|          | 0/217 [00:00<?, ?it/s]          /Users/yuen@backbase.com/Documents/Baddiev2/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
[A10_bones_bbox] Epoch 33/80:   0%|          | 0/217 [00:00<?, ?it/s]          /Users/yuen@backbase.com/Documents/Baddiev2/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)

  Early stopping at epoch 36 (best val F1: 0.5206)


/Users/yuen@backbase.com/Documents/Baddiev2/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)



  === A10_bones_bbox ====
  Test Accuracy:  0.5925
  Test Macro-F1:  0.4563
  Top-3 Accuracy: 0.8728
  Top-5 Accuracy: 0.9275
  Best Val F1:    0.5206
  Epochs:         36
  Time:           35515s
                        precision    recall  f1-score   support

              net shot       0.82      0.81      0.82       296
            return net       0.75      0.62      0.68       163
                 smash       0.51      0.24      0.33       107
           wrist smash       0.31      0.61      0.41        75
                   lob       0.39      0.89      0.54       137
  defensive return lob       0.31      0.50      0.38         8
                 clear       0.86      0.88      0.87       129
                 drive       0.46      0.11      0.18        53
         driven flight       0.00      0.00      0.00         1
      back-court drive       0.75      0.12      0.21        25
                  drop       0.32      0.74      0.45        61
          passive drop       0.59

In [ ]:
# ── Group A+ Summary ──────────────────────────────────────────────────────
print('\n' + '='*70)
print('GROUP A+: Feature Representation (no shuttle, mean pooling)')
print('='*70)
a_results = [result_A7, result_A8, result_A9, result_A10]
print(f'{"Ablation":<30} {"Acc":>8} {"F1":>8} {"Top-3":>8} {"Top-5":>8}')
print('-'*70)
for r in a_results:
    print(f'{r["name"]:<30} {r["accuracy"]:>8.4f} {r["macro_f1"]:>8.4f} '
          f'{r["top3_acc"]:>8.4f} {r["top5_acc"]:>8.4f}')

best_a = max(a_results, key=lambda r: r['macro_f1'])
print(f'\nWinner: {best_a["name"]} (F1={best_a["macro_f1"]:.4f})')
print(f'Run4 A1 reference: F1=0.5461, Acc=0.6060')

## 4. Group B+: Pooling Strategy (builds on A+ winner)

Tests attention pooling and max pooling against mean pooling.
Uses the A+ winner's feature config, **still no shuttle** — isolating pooling effect.

| Ablation | Pooling | Description |
|----------|---------|-------------|
| B+ winner from A+ | mean | Baseline (A+ winner already uses mean pooling) |
| B3 | attn | Learned temporal + joint attention weights |
| B4 | max | Global max pooling (preserves peak activations) |

In [ ]:
# Determine A+ winner config
_a_winner = best_a['name']
_use_bones = 'bones' in _a_winner or 'A8' in _a_winner or 'A10' in _a_winner
_use_bbox  = 'bbox' in _a_winner or 'A9' in _a_winner or 'A10' in _a_winner
_in_ch = FEATURE_DIMS['L2'] + (BONE_CHANNELS if _use_bones else 0)
print(f'Group B+ uses A+ winner config: bones={_use_bones}, bbox_norm={_use_bbox}, in_ch={_in_ch}')

In [ ]:
# ── B3: Attention pooling ─────────────────────────────────────────────────
print('B3: Attention pooling + A+ winner features (no shuttle)')
train_B3 = make_dataset(TRAIN_MATCHES, feature_layer='L2',
                        use_bones=_use_bones, use_bbox_norm=_use_bbox)
val_B3   = make_dataset(VAL_MATCHES, feature_layer='L2',
                        use_bones=_use_bones, use_bbox_norm=_use_bbox)
test_B3  = make_dataset(TEST_MATCHES, feature_layer='L2',
                        use_bones=_use_bones, use_bbox_norm=_use_bbox)

encoder_B3 = build_encoder(in_channels=_in_ch, num_nodes=NUM_NODES, pooling='attn')
head_B3 = nn.Linear(256, N_CLASSES).to(device)
print(f'Parameters: {sum(p.numel() for p in encoder_B3.parameters()) + sum(p.numel() for p in head_B3.parameters()):,}')

result_B3 = train_and_evaluate('B3_attn_pool', train_B3, val_B3, test_B3,
                                encoder_B3, head_B3)

In [ ]:
# ── B4: Max pooling ───────────────────────────────────────────────────────
print('B4: Max pooling + A+ winner features (no shuttle)')
train_B4 = make_dataset(TRAIN_MATCHES, feature_layer='L2',
                        use_bones=_use_bones, use_bbox_norm=_use_bbox)
val_B4   = make_dataset(VAL_MATCHES, feature_layer='L2',
                        use_bones=_use_bones, use_bbox_norm=_use_bbox)
test_B4  = make_dataset(TEST_MATCHES, feature_layer='L2',
                        use_bones=_use_bones, use_bbox_norm=_use_bbox)

encoder_B4 = build_encoder(in_channels=_in_ch, num_nodes=NUM_NODES, pooling='max')
head_B4 = nn.Linear(256, N_CLASSES).to(device)
print(f'Parameters: {sum(p.numel() for p in encoder_B4.parameters()) + sum(p.numel() for p in head_B4.parameters()):,}')

result_B4 = train_and_evaluate('B4_max_pool', train_B4, val_B4, test_B4,
                                encoder_B4, head_B4)

In [ ]:
# ── Group B+ Summary ──────────────────────────────────────────────────────
print('\n' + '='*70)
print('GROUP B+: Pooling Strategy (no shuttle)')
print('='*70)
b_results = [best_a, result_B3, result_B4]
print(f'{"Ablation":<30} {"Acc":>8} {"F1":>8} {"Top-3":>8} {"Top-5":>8} {"Pooling":<10}')
print('-'*70)
pooling_labels = ['mean', 'attn', 'max']
for r, pl in zip(b_results, pooling_labels):
    print(f'{r["name"]:<30} {r["accuracy"]:>8.4f} {r["macro_f1"]:>8.4f} '
          f'{r["top3_acc"]:>8.4f} {r["top5_acc"]:>8.4f} {pl:<10}')

best_b = max(b_results, key=lambda r: r['macro_f1'])
_best_pooling = 'attn' if 'B3' in best_b['name'] else ('max' if 'B4' in best_b['name'] else 'mean')
print(f'\nWinner: {best_b["name"]} (F1={best_b["macro_f1"]:.4f}, pooling={_best_pooling})')

## 5. Group C+: Full Cascade (A+ winner + B+ winner + shuttle cross-attention)

Re-run shuttle cross-attention with the improved encoder from A+ and B+.
This is the equivalent of run4's C3, but with better features and pooling.

In [ ]:
# ── C4: Full cascade ──────────────────────────────────────────────────────
print(f'C4: A+ winner ({_a_winner}) + B+ winner ({_best_pooling} pool) + shuttle cross-attn')
train_C4 = make_dataset(TRAIN_MATCHES, feature_layer='L2',
                        use_shuttle=True, shuttle_fusion='cross_attn',
                        use_bones=_use_bones, use_bbox_norm=_use_bbox)
val_C4   = make_dataset(VAL_MATCHES, feature_layer='L2',
                        use_shuttle=True, shuttle_fusion='cross_attn',
                        use_bones=_use_bones, use_bbox_norm=_use_bbox)
test_C4  = make_dataset(TEST_MATCHES, feature_layer='L2',
                        use_shuttle=True, shuttle_fusion='cross_attn',
                        use_bones=_use_bones, use_bbox_norm=_use_bbox)

encoder_C4 = build_encoder(in_channels=_in_ch, num_nodes=NUM_NODES, pooling=_best_pooling)
head_C4 = nn.Linear(256, N_CLASSES).to(device)
ca_C4 = ShuttleCrossAttention(d_skel=256, d_shuttle=128, nhead=4).to(device)
total_params = (sum(p.numel() for p in encoder_C4.parameters()) +
                sum(p.numel() for p in head_C4.parameters()) +
                sum(p.numel() for p in ca_C4.parameters()))
print(f'Parameters: {total_params:,}')

result_C4 = train_and_evaluate('C4_full_cascade', train_C4, val_C4, test_C4,
                                encoder_C4, head_C4, cross_attn_module=ca_C4)

## 6. Overall Summary

In [ ]:
all_results = [result_A7, result_A8, result_A9, result_A10,
               result_B3, result_B4, result_C4]

print('\n' + '='*80)
print('RUN 5 — FULL RESULTS')
print('='*80)
print(f'Pipeline: A1 → A+(bones/bbox) → B+(pooling) → C+(shuttle cross-attn)')
print()
print(f'{"":<4} {"Ablation":<30} {"Acc":>8} {"F1":>8} {"Top-3":>8} {"Top-5":>8} {"Epochs":>8}')
print('-'*80)

best_overall = max(all_results, key=lambda r: r['macro_f1'])
for i, r in enumerate(all_results):
    marker = ' <-- best' if r['name'] == best_overall['name'] else ''
    print(f'{i:<4} {r["name"]:<30} {r["accuracy"]:>8.4f} {r["macro_f1"]:>8.4f} '
          f'{r["top3_acc"]:>8.4f} {r["top5_acc"]:>8.4f} {r["stopped_epoch"]:>8}{marker}')

# Run 4 references
print(f'\n--- Run 4 references ---')
print(f'     A1 (dual L2, no shuttle):     Acc=0.6060  F1=0.5461')
print(f'     C3 (+ shuttle cross-attn):    Acc=0.6189  F1=0.5929')

print(f'\n--- Run 5 best ---')
print(f'     {best_overall["name"]}:')
print(f'       Macro-F1:  {best_overall["macro_f1"]:.4f}  (delta vs C3: {best_overall["macro_f1"] - 0.5929:+.4f})')
print(f'       Accuracy:  {best_overall["accuracy"]:.4f}  (delta vs C3: {best_overall["accuracy"] - 0.6189:+.4f})')
print(f'       Top-3 Acc: {best_overall["top3_acc"]:.4f}')

## 7. Per-Class Diagnostic (Best Config vs Run 4 C3)

In [ ]:
best_name = best_overall['name']
print(f'Detailed diagnostics for: {best_name}\n')

per_class = best_overall['per_class']

# Run4 C3 per-class F1 values for comparison
r4_c3_f1s = {
    'short_serve': 0.90, 'long_serve': 0.89, 'smash': 0.40, 'tap_smash': 0.43,
    'push_rush': 0.46, 'clear': 0.92, 'slice_drop': 0.48, 'net_drop': 0.76,
    'transition': 0.59, 'drive': 0.34, 'block': 0.65, 'lob_lift': 0.61,
    'defensive_lift': 0.53, 'cross_net': 0.47, 'push': 0.49,
}

print(f'{"Class":<20} {"Run4 C3":>10} {"Run5 Best":>10} {"Delta":>10} {"Support":>10}')
print('-'*65)
for cls in SHOT_TYPES:
    r4 = r4_c3_f1s.get(cls, 0)
    r5_data = per_class.get(cls, {})
    r5 = r5_data.get('f1-score', 0)
    support = r5_data.get('support', 0)
    delta = r5 - r4
    flag = ' !!!' if abs(delta) > 0.05 else ''
    if support > 0:
        print(f'{cls:<20} {r4:>10.3f} {r5:>10.3f} {delta:>+10.3f} {support:>10.0f}{flag}')

## 8. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for r in all_results:
    axes[0].plot(r['history']['train_loss'], label=r['name'], alpha=0.8)
    axes[1].plot(r['history']['val_f1'], label=r['name'], alpha=0.8)

axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Train Loss'); axes[0].set_title('Training Loss')
axes[0].legend(fontsize=7); axes[0].set_yscale('log')

axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Val Macro-F1'); axes[1].set_title('Validation F1')
axes[1].legend(fontsize=7)
axes[1].axhline(y=0.5442, color='gray', linestyle='--', alpha=0.5, label='Run4 A1 val F1')
axes[1].axhline(y=0.5769, color='orange', linestyle='--', alpha=0.5, label='Run4 C3 val F1')

plt.tight_layout()
plt.savefig(ABLATION_DIR / 'run5_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {ABLATION_DIR / "run5_training_curves.png"}')